<!-- Curated copy -->
> **Curated copy.** This notebook is taken verbatim from the BTech-thesis working archive; only
> cell *outputs* have been cleared and machine-specific absolute paths (`C:\\...`, `D:\\...`,
> `F:\\...`) have been rewritten to repository-relative `runs/...` paths. No scientific logic,
> equation, hyper-parameter or architecture has been modified. Place regenerated
> `dataset_run_*` folders under a `runs/` directory next to this notebook (or edit the paths).
> The figures this notebook originally produced are preserved in the sibling `figures/` folder.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

import os, json,ast, math, glob, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import glob
import pandas as pd
from sklearn.decomposition import PCA

In [ ]:
N        = 120          # grid points in x,y (Q, T, f are NxN)
Lx, Ly   = 0.05, 0.05
rho, cp, k, L_lat = 800.0, 2000.0, 0.2, 2e5
T_m      = 330.0
T_init   = 300.0
T_bound  = 330.0
t_end    = 4000.0
save_times = (100.0, 250.0, 400.0, 600.0, 1000.0, 1200.0, 1500.0, 1800.0, 2100.0)
cfl      = 0.45

# Heat-source GP
q_scale        = 1e5     # W/m^3
Q_length_scale = 0.18
Q_sigma        = 1.0

# Boundary Conditions
mu_max=0.8
sigma_max=0.7
amp_max=80

# Dataset sizes
NUM_CASES      = 500.0      # total function-realizations / cases
TRAIN_FRAC     = 0.8
VAL_FRAC       = 0.1      # test is the remainder

# DeepONet sampling
sensor_mode    = "full"   # "full" (flatten full NxN) or "downsample"
S_down         = 40       # used if sensor_mode="downsample" (S_down x S_down grid)
n_time_bc      = len(save_times)
S_down_BC      = N

points_per_case_per_time = N**2  # # of (x,y) points sampled per time snapshot

# Boundary configuration control
only_lr_vary   = True     # True: top & bottom constant, left/right varied; False: allow all configurable
All_side_const_temp_boundary = False    # if all sides constant boundary is wanted at T_bound for dataset testing

# MODEL PARAMETERS

BATCH_POINTS = 32768//2
EPOCHS = 60
LR = 1e-4
WEIGHT_DECAY = 1e-4# first case 0.0
VAL_SAMPLES = 4
VAL_BATCH_POINTS = 32768//2
STEPS_PER_EPOCH = 64
NUM_WORKERS = 0
BATCH_SIZE = 16
ACTIVATION_FUNCTION="sin"
TARGET = "T"          # "T" for temperature, "f" for liquid fraction


# Paper-like feature size: f=64 for one task, f=96 for the other. Start with 64.
F_WIDTH = 64
TRUNK_HIDDEN = 128
TRUNK_DEPTH = 10      # paper baseline says 10

# DeepONet path-weights: weights for the additive and multiplicative
#each path weight
alpha_add=1.0
alpha_prod=0.2

#U-Net parameters
unet_dropout=0.0
unet_features=64

In [ ]:
import glob
# -----------------------------
# 0) Config / Auto-run directory
# -----------------------------
ROOT = Path(".")
# hardcoded one: RUN_DIR = Path("./dataset_run_YYYYMMDD-HHMMSS")
def _latest_run_dir(root: Path) -> Path | None:
    cands = sorted([Path(p) for p in glob.glob(str(root / "dataset_run_*")) if Path(p).is_dir()])
    return cands[-1] if cands else None

RUN_DIR = _latest_run_dir(ROOT)
assert RUN_DIR is not None, "No dataset_run_* folder found. Build dataset first."

print(f"Using RUN_DIR: {RUN_DIR}")

NPZ_NAME = "udeeponet_dataset.npz"
SPLITS_NAME = "splits.json"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

In [ ]:
npz_path = RUN_DIR / NPZ_NAME
splits_path = RUN_DIR / SPLITS_NAME

assert npz_path.exists(), f"Missing: {npz_path}"
assert splits_path.exists(), f"Missing: {splits_path}"

print("Found dataset:", npz_path.resolve())
print("Found splits :", splits_path.resolve())


In [ ]:
'''npz = np.load(npz_path, allow_pickle=True)

X_static_np = npz["X_static"].astype(np.float32)   # [C, Cin, H, W]
times_np    = npz["times"].astype(np.float32)      # [T]
Y_T_np      = npz["Y_T"].astype(np.float32)        # [C, T, H, W]
Y_f_np      = npz["Y_f"].astype(np.float32)        # [C, T, H, W]
case_ids_np = npz["case_ids"].astype(np.int64)

# ---- robust meta load: works whether meta is dict, JSON string, or np scalar ----
meta_raw = npz["meta"]
meta_raw = meta_raw.item() if isinstance(meta_raw, np.ndarray) else meta_raw

if isinstance(meta_raw, (bytes, np.bytes_)):
    meta_raw = meta_raw.decode("utf-8")

if isinstance(meta_raw, str):
    meta = json.loads(meta_raw)   # string -> dict
elif isinstance(meta_raw, dict):
    meta = meta_raw               # already dict
else:
    # fallback: try converting numpy object to python object then parse if string
    meta_raw = meta_raw.item() if hasattr(meta_raw, "item") else meta_raw
    meta = json.loads(meta_raw) if isinstance(meta_raw, str) else dict(meta_raw)

C, Cin, H, W = X_static_np.shape
T = times_np.shape[0]

print("X_static:", X_static_np.shape)
print("times   :", times_np.shape, times_np[:5], "...")
print("Y_T     :", Y_T_np.shape)
print("Y_f     :", Y_f_np.shape)
print("Cin(meta):", meta.get("Cin"), " pad:", meta.get("pad_h"), meta.get("pad_w"))'''


In [ ]:
npz = np.load(npz_path, allow_pickle=True)
X_static_np = npz["X_static"].astype(np.float32)   # [C, Cin, H, W]
times_np    = npz["times"].astype(np.float32)      # [T]
Y_T_np      = npz["Y_T"].astype(np.float32)        # [C, T, H, W]
Y_f_np      = npz["Y_f"].astype(np.float32)        # [C, T, H, W]
case_ids_np = npz["case_ids"].astype(np.int64)
meta        = npz["meta"].item()

C, Cin, H, W = X_static_np.shape
T = times_np.shape[0]

print("X_static:", X_static_np.shape)
print("times   :", times_np.shape, times_np[:5], "...")
print("Y_T     :", Y_T_np.shape)
print("Y_f     :", Y_f_np.shape)
print("Cin(meta):", meta.get("Cin"), " pad:", meta.get("pad_h"), meta.get("pad_w"))



In [ ]:
with open(splits_path, "r") as f:
    splits = json.load(f)

train_ids = np.array(splits["train"], dtype=np.int64)
val_ids   = np.array(splits["val"], dtype=np.int64)
test_ids  = np.array(splits["test"], dtype=np.int64)

print("Split sizes:", len(train_ids), len(val_ids), len(test_ids))


In [ ]:
def compute_train_stats(X_static_np, Y_np, train_ids):
    """
    X_static_np: [C,Cin,H,W]
    Y_np:        [C,T,H,W]  (temperature)
    train_ids: list[int]
    Returns dict of numpy arrays for mean/std.
    """
    Xtr = X_static_np[train_ids]  # [Ctr,Cin,H,W]
    Ytr = Y_np[train_ids]         # [Ctr,T,H,W]

    # per-channel mean/std over (case, H, W)
    x_mean = Xtr.mean(axis=(0,2,3), keepdims=True)  # [1,Cin,1,1]
    x_std  = Xtr.std(axis=(0,2,3), keepdims=True) + 1e-6

    # scalar mean/std for Y over everything
    y_mean = Ytr.mean()
    y_std  = Ytr.std() + 1e-6

    return {
        "x_mean": x_mean.astype(np.float32),
        "x_std":  x_std.astype(np.float32),
        "y_mean": float(y_mean),
        "y_std":  float(y_std),
    }

Y_target_np = Y_T_np if TARGET == "T" else Y_f_np
stats = compute_train_stats(X_static_np, Y_target_np, train_ids)
print("Y mean/std:", stats["y_mean"], stats["y_std"])
print("X mean/std shapes:", stats["x_mean"].shape, stats["x_std"].shape)


In [ ]:
class UDeepONetCaseDataset(Dataset):
    """
    Returns full-grid sample per case:
      X_static: [Cin,H,W]  (normalized)
      times:    [T]        (normalized to [0,1])
      Y:        [T,H,W]    (normalized)
    """
    def __init__(self, X_static, times, Y_T, Y_f, ids, stats, target="T"):
        self.X_static = X_static
        self.times = times
        self.Y_T = Y_T
        self.Y_f = Y_f
        self.ids = ids
        self.stats = stats
        assert target in ["T", "f"]
        self.target = target

        self.t_scale = float(np.max(times)) if float(np.max(times)) != 0.0 else 1.0

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        c = self.ids[i]

        X = self.X_static[c].astype(np.float32)  # [Cin,H,W]
        # normalize X per channel
        X = (X - self.stats["x_mean"][0]) / self.stats["x_std"][0]

        t = (self.times.astype(np.float32) / self.t_scale)  # [T] in [0,1]

        if self.target == "T":
            Y = self.Y_T[c].astype(np.float32)  # [T,H,W]
        else:
            Y = self.Y_f[c].astype(np.float32)

        # normalize Y (scalar)
        Y = (Y - self.stats["y_mean"]) / self.stats["y_std"]

        return {
            "X_static": torch.from_numpy(X),
            "times": torch.from_numpy(t),
            "Y": torch.from_numpy(Y),
            "case_index": int(c),
            "Y_T": torch.from_numpy(self.Y_T[c].astype(np.float32)),
            "Y_f": torch.from_numpy(self.Y_f[c].astype(np.float32)),
        }


In [ ]:
ds_train = UDeepONetCaseDataset(X_static_np, times_np, Y_T_np, Y_f_np, train_ids,stats=stats, target=TARGET)
ds_val   = UDeepONetCaseDataset(X_static_np, times_np, Y_T_np, Y_f_np, val_ids,  stats=stats, target=TARGET)

dl_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=NUM_WORKERS, pin_memory=True)
dl_val   = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)

batch = next(iter(dl_train))
print("Batch shapes:",
      batch["X_static"].shape,  # [B,Cin,H,W]
      batch["times"].shape,     # [B,T]
      batch["Y"].shape)         # [B,T,H,W]


In [ ]:
class PaperUNetBlock2D(nn.Module):
    """
    Paper Table-2 style U-Net block, channel width fixed = f.
    """
    def __init__(self, f: int):
        super().__init__()
        act = lambda: nn.LeakyReLU(0.2, inplace=True)

        def conv3(in_ch, out_ch, stride):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),
                act()
            )

        def deconv4(in_ch, out_ch):
            return nn.Sequential(
                nn.ConvTranspose2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1, bias=True),
                act()
            )

        self.c1 = conv3(f, f, stride=2)   # stage1
        self.c2 = conv3(f, f, stride=2)   # stage2
        self.c3 = conv3(f, f, stride=1)   # stage3
        self.c4 = conv3(f, f, stride=2)   # bottleneck
        self.c5 = conv3(f, f, stride=1)   # bottleneck

        self.u1 = deconv4(f,   f)         # up to stage3 size
        self.u2 = deconv4(2*f, f)         # up to stage1 size
        self.u3 = deconv4(2*f, f)         # up to input size

        self.out = nn.Conv2d(2*f, f, kernel_size=3, stride=1, padding=1, bias=True)

    def forward(self, x):
        x_in = x
        s1 = self.c1(x_in)
        s2 = self.c2(s1)
        s3 = self.c3(s2)
        b  = self.c4(s3)
        b  = self.c5(b)

        x = self.u1(b)
        x = torch.cat([x, s3], dim=1)

        x = self.u2(x)
        x = torch.cat([x, s1], dim=1)

        x = self.u3(x)
        x = torch.cat([x, x_in], dim=1)

        x = self.out(x)
        return x


class TrunkTimeMLP(nn.Module):
    """
    Time-only trunk, 10-layer baseline.
    """
    def __init__(self, f: int, hidden: int = 128, depth: int = 10, act=ACTIVATION_FUNCTION):
        super().__init__()
        assert depth >= 2
        self.act = act
        layers = []
        in_dim = 1
        for i in range(depth - 1):
            out_dim = hidden if i < depth - 2 else f
            layers.append(nn.Linear(in_dim, out_dim))
            in_dim = out_dim
        self.layers = nn.ModuleList(layers)

    def forward(self, t):
        if t.ndim == 1:
            t = t[:, None]
        x = t
        for i, lin in enumerate(self.layers):
            x = lin(x)
            if i < len(self.layers) - 1:
                if self.act == "sin":
                    x = torch.sin(x)
                else:
                    x = F.relu(x)
        return x  # [T,f]


class UDeepONet_Paper(nn.Module):
    """
    X_static: [B,Cin,H,W]
    times:    [T]  (same for entire batch)
    output:   [B,T,H,W]
    """
    def __init__(self, Cin: int, f: int = 64, trunk_hidden: int = 128, trunk_depth: int = 10, act=ACTIVATION_FUNCTION):
        super().__init__()
        self.f = f
        self.lift = nn.Conv2d(Cin, f, kernel_size=1, bias=True)

        self.unet1 = PaperUNetBlock2D(f)
        self.unet2 = PaperUNetBlock2D(f)
        self.unet3 = PaperUNetBlock2D(f)
        self.sigma = nn.LeakyReLU(0.2, inplace=True)

        self.trunk = TrunkTimeMLP(f=f, hidden=trunk_hidden, depth=trunk_depth, act=act)

        self.proj1 = nn.Conv3d(f, f, kernel_size=1, bias=True)
        self.proj2 = nn.Conv3d(f, 1, kernel_size=1, bias=True)

    def forward(self, X_static, times):
        # Branch
        x = self.lift(X_static)         # [B,f,H,W]
        x = self.unet1(x); x = self.sigma(x)
        x = self.unet2(x); x = self.sigma(x)
        x = self.unet3(x)               # [B,f,H,W]

        # Trunk
        t_feat = self.trunk(times)      # [T,f]

        # Merge: [B,T,f,H,W]
        xt = x[:, None, :, :, :] * t_feat[None, :, :, None, None]

        # Project with 1x1x1 convs: [B,f,T,H,W] -> [B,1,T,H,W] -> [B,T,H,W]
        xt = xt.permute(0, 2, 1, 3, 4).contiguous()
        y = F.relu(self.proj1(xt))
        y = self.proj2(y)               # [B,1,T,H,W]
        y = y[:, 0]                     # [B,T,H,W]
        return y


In [ ]:
model = UDeepONet_Paper(Cin=Cin, f=F_WIDTH, trunk_hidden=TRUNK_HIDDEN, trunk_depth=TRUNK_DEPTH, act=ACTIVATION_FUNCTION).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)

# simple MSE
def mse_loss(pred, target):
    return torch.mean((pred - target)**2)

# RMSE helper (nice to log)
def rmse(pred, target):
    return torch.sqrt(torch.mean((pred - target)**2) + 1e-12)


In [ ]:
import torch, importlib
print(torch.__version__)
importlib.import_module("torch._utils")
import torch.optim as optim
optim.Adam([torch.nn.Parameter(torch.randn(2,requires_grad=True))], lr=1e-3)
print("Adam OK")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device used: {device}")
print("\n")


In [ ]:
'''def run_epoch(model, loader, train: bool):
    model.train(train)
    total_loss = 0.0
    total_rmse = 0.0
    n = 0

    for batch in loader:
        X = batch["X_static"].to(DEVICE, non_blocking=True)   # [B,Cin,H,W]
        Y = batch["Y"].to(DEVICE, non_blocking=True)          # [B,T,H,W]

        # times is identical across the batch, take first row
        times = batch["times"][0].to(DEVICE)                  # [T]

        if train:
            opt.zero_grad(set_to_none=True)

        pred = model(X, times)                                # [B,T,H,W]
        loss = mse_loss(pred, Y)

        if train:
            loss.backward()
            opt.step()

        with torch.no_grad():
            r = rmse(pred, Y)

        bs = X.shape[0]
        total_loss += float(loss) * bs
        total_rmse += float(r) * bs
        n += bs

    return total_loss / max(n,1), total_rmse / max(n,1)'''


In [ ]:
def run_epoch(model, loader, train: bool, epoch=None):
    model.train(train)

    total_loss = 0.0
    total_rmse = 0.0
    n = 0

    num_batches = len(loader)

    for iteration, batch in enumerate(loader, start=1):

        X = batch["X_static"].to(DEVICE, non_blocking=True)   # [B,Cin,H,W]
        Y = batch["Y"].to(DEVICE, non_blocking=True)          # [B,T,H,W]

        # times is identical across the batch, take first row
        times = batch["times"][0].to(DEVICE)                  # [T]

        if train:
            opt.zero_grad(set_to_none=True)

        pred = model(X, times)                                # [B,T,H,W]
        loss = mse_loss(pred, Y)

        if train:
            loss.backward()
            opt.step()

        with torch.no_grad():
            r = rmse(pred, Y)

        bs = X.shape[0]

        total_loss += float(loss) * bs
        total_rmse += float(r) * bs
        n += bs

        # Running averages
        avg_loss = total_loss / max(n, 1)
        avg_rmse = total_rmse / max(n, 1)

        phase = "TRAIN" if train else "VAL"

        print(
            f"Epoch {epoch:03d} | "
            f"{phase} | "
            f"Iter {iteration:03d}/{num_batches:03d} | "
            f"Loss {float(loss):.4e} | "
            f"RMSE {float(r):.4e} | "
            f"Avg Loss {avg_loss:.4e} | "
            f"Avg RMSE {avg_rmse:.4e}"
        )

    return total_loss / max(n, 1), total_rmse / max(n, 1)

In [ ]:
'''ckpt_dir = RUN_DIR / f"checkpoints_udeeponet_{TARGET}"
ckpt_dir.mkdir(parents=True, exist_ok=True)

best_val = float("inf")

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    tr_loss, tr_rmse = run_epoch(model, dl_train, train=True)
    va_loss, va_rmse = run_epoch(model, dl_val,   train=False)

    dt = time.time() - t0
    print(f"epoch {epoch:03d} | "
          f"train loss {tr_loss:.4e} rmse {tr_rmse:.4e} | "
          f"val loss {va_loss:.4e} rmse {va_rmse:.4e} | "
          f"{dt:.1f}s")

    # save best
    if va_loss < best_val:
        best_val = va_loss
        ckpt_path = ckpt_dir / "best.pt"
        torch.save({
            "model": model.state_dict(),
            "opt": opt.state_dict(),
            "epoch": epoch,
            "best_val": best_val,
            "meta": meta,
            "Cin": Cin,
            "f": F_WIDTH,
            "trunk_hidden": TRUNK_HIDDEN,
            "trunk_depth": TRUNK_DEPTH,
            "target": TARGET,
        }, ckpt_path)
        print("  saved:", ckpt_path)'''


In [ ]:
ckpt_dir = RUN_DIR / f"checkpoints_udeeponet_{TARGET}"
ckpt_dir.mkdir(parents=True, exist_ok=True)

best_val = float("inf")

for epoch in range(1, EPOCHS + 1):

    t0 = time.time()

    tr_loss, tr_rmse = run_epoch(
        model,
        dl_train,
        train=True,
        epoch=epoch
    )

    va_loss, va_rmse = run_epoch(
        model,
        dl_val,
        train=False,
        epoch=epoch
    )

    dt = time.time() - t0

    print("\n" + "=" * 90)
    print(
        f"Epoch {epoch:03d} COMPLETE | "
        f"Train Loss {tr_loss:.4e} | "
        f"Train RMSE {tr_rmse:.4e} | "
        f"Val Loss {va_loss:.4e} | "
        f"Val RMSE {va_rmse:.4e} | "
        f"Time {dt:.1f}s"
    )
    print("=" * 90)

    # Save best model
    if va_loss < best_val:

        best_val = va_loss

        ckpt_path = ckpt_dir / "best.pt"

        torch.save({
            "model": model.state_dict(),
            "opt": opt.state_dict(),
            "epoch": epoch,
            "best_val": best_val,
            "meta": meta,
            "Cin": Cin,
            "f": F_WIDTH,
            "trunk_hidden": TRUNK_HIDDEN,
            "trunk_depth": TRUNK_DEPTH,
            "target": TARGET,
        }, ckpt_path)

        print(f"  ✓ Best model saved: {ckpt_path}")

In [ ]:
@torch.no_grad()
def plot_heat_trueT_predT(
    model,
    dataset,
    stats_T,
    case_idx_in_dataset=0,
    ncols=5,
    meta=None,
    heat_ch=0,
    q_scale=1.0,               # set to 1e5 if channel stores a dimensionless field
    heat_unit=r"W/m$^3$",
    cmap_heat="Reds",
    cmap_T="inferno",
    show_colorbar=True,
):
    """
    Rows:
      0 : Heat source (static, TRUE physical values)
      1 : True Temperature
      2 : Predicted Temperature

    Notes:
    - True and predicted temperature share one color scale so they are
      directly comparable.
    - The heat source is shown in its original (un-normalized) units.
    """

    model.eval()

    sample = dataset[case_idx_in_dataset]

    # ----------------------------
    # Inputs
    # ----------------------------
    Xn = sample["X_static"][None].to(DEVICE)
    t = sample["times"].to(DEVICE)

    # ----------------------------
    # Ground truth temperature (raw, K)
    # ----------------------------
    YT_true = sample["Y_T"].cpu().numpy()

    # ----------------------------
    # Prediction
    # ----------------------------
    predT_n = model(Xn, t)[0].cpu().numpy()

    # Denormalize temperature
    predT = predT_n * stats_T["y_std"] + stats_T["y_mean"]

    # ----------------------------
    # Time indices to display
    # ----------------------------
    Tn = predT.shape[0]
    idxs = np.linspace(0, Tn - 1, min(ncols, Tn)).round().astype(int)

    # ----------------------------
    # Physical extent
    # ----------------------------
    extent = None
    if meta is not None and ("Lx" in meta) and ("Ly" in meta):
        extent = [
            0,
            float(meta["Lx"]),
            0,
            float(meta["Ly"]),
        ]

    # ----------------------------
    # Physical times
    # ----------------------------
    times_phys = getattr(dataset, "times_raw", None)
    if times_phys is None:
        times_phys = sample["times"].cpu().numpy()

    # ----------------------------
    # Heat source in TRUE (physical) units
    # ----------------------------
    # sample["X_static"] is normalized: (X - x_mean) / x_std.
    # Prefer the raw array the dataset holds; else invert the normalization.

    X_static = sample["X_static"].cpu().numpy()

    if heat_ch < 0 or heat_ch >= X_static.shape[0]:
        raise ValueError(
            f"heat_ch={heat_ch} out of range for Cin={X_static.shape[0]}"
        )

    case_id = int(sample["case_index"])

    Qmap = None
    X_raw_all = getattr(dataset, "X_static", None)

    if X_raw_all is not None:
        try:
            Qmap = np.asarray(X_raw_all[case_id][heat_ch], dtype=np.float64)
        except Exception:
            Qmap = None

    if Qmap is None:
        st = getattr(dataset, "stats", None)
        if st is None:
            raise RuntimeError(
                "Cannot recover the true heat source: dataset has neither "
                "a raw X_static array nor normalization stats."
            )
        x_mean = np.asarray(st["x_mean"], dtype=np.float64).reshape(-1)[heat_ch]
        x_std = np.asarray(st["x_std"], dtype=np.float64).reshape(-1)[heat_ch]
        Qmap = X_static[heat_ch].astype(np.float64) * x_std + x_mean

    Qmap = Qmap * float(q_scale)

    # =====================================================
    # UNIFORM COLOR SCALES
    # =====================================================

    # ---------------- Heat source (true units) ----------------
    q_vmin = float(Qmap.min())
    q_vmax = float(Qmap.max())

    if np.isclose(q_vmin, q_vmax):
        q_vmax = q_vmin + 1e-8

    # ---------------- Temperature (shared true/pred scale) ----------------
    T_vmin = float(min(YT_true.min(), predT.min()))
    T_vmax = float(max(YT_true.max(), predT.max()))

    if np.isclose(T_vmin, T_vmax):
        T_vmax = T_vmin + 1e-8

    # =====================================================
    # Figure
    # =====================================================

    fig, axs = plt.subplots(
        3,
        len(idxs),
        figsize=(3 * len(idxs), 8),
        constrained_layout=True,
    )

    if len(idxs) == 1:
        axs = axs.reshape(3, 1)

    im_heat = None
    im_trueT = None
    im_predT = None

    # =====================================================
    # Plot
    # =====================================================

    for j, ti in enumerate(idxs):

        # ---------------- Row 0: Heat source ----------------

        ax = axs[0, j]

        im_heat = ax.imshow(
            Qmap.T,
            origin="lower",
            cmap=cmap_heat,
            vmin=q_vmin,
            vmax=q_vmax,
            extent=extent,
            interpolation="nearest",
        )

        ax.set_title(f"Heat Source (peak {q_vmax:.3g})")
        ax.axis("off")

        # ---------------- Row 1: True Temperature ----------------

        ax = axs[1, j]

        im_trueT = ax.imshow(
            YT_true[ti].T,
            origin="lower",
            cmap=cmap_T,
            vmin=T_vmin,
            vmax=T_vmax,
            extent=extent,
            interpolation="nearest",
        )

        ax.set_title(f"True T ({times_phys[ti]:.0f} s)")
        ax.axis("off")

        # ---------------- Row 2: Predicted Temperature ----------------

        ax = axs[2, j]

        im_predT = ax.imshow(
            predT[ti].T,
            origin="lower",
            cmap=cmap_T,
            vmin=T_vmin,
            vmax=T_vmax,
            extent=extent,
            interpolation="nearest",
        )

        ax.set_title(f"Pred T ({times_phys[ti]:.0f} s)")
        ax.axis("off")

    # =====================================================
    # Colorbars
    # =====================================================

    if show_colorbar:

        from matplotlib.ticker import ScalarFormatter

        # ---------------- Heat source ----------------

        cbar = fig.colorbar(
            im_heat,
            ax=axs[0, :],
            fraction=0.025,
            pad=0.02,
        )
        cbar.set_label(f"Heat Source, q ({heat_unit})")

        _fmt = ScalarFormatter(useMathText=True)
        _fmt.set_powerlimits((-2, 3))
        cbar.formatter = _fmt
        cbar.update_ticks()

        # ---------------- True temperature ----------------

        cbar = fig.colorbar(
            im_trueT,
            ax=axs[1, :],
            fraction=0.025,
            pad=0.02,
        )
        cbar.set_label("Temperature (K)")

        # ---------------- Predicted temperature ----------------

        cbar = fig.colorbar(
            im_predT,
            ax=axs[2, :],
            fraction=0.025,
            pad=0.02,
        )
        cbar.set_label("Temperature (K)")

    plt.show()


def load_model_from_ckpt(
    ckpt_path: str | Path,
    Cin: int | None = None,
):
    """
    Loads your training checkpoint dict that stores:
    ck['model'] = state_dict, plus ck['Cin'], ck['f'], ck['trunk_hidden'], ck['trunk_depth'].
    """
    ckpt_path = Path(ckpt_path)
    ck = torch.load(ckpt_path, map_location=DEVICE)

    # architecture info (prefer checkpoint values)
    Cin = ck.get("Cin", Cin)
    f = ck.get("f")
    trunk_hidden = ck.get("trunk_hidden")
    trunk_depth = ck.get("trunk_depth")

    assert Cin is not None, "Cin must be provided or stored in checkpoint"
    assert f is not None and trunk_hidden is not None and trunk_depth is not None, "Checkpoint missing arch fields"

    model = UDeepONet_Paper(
        Cin=Cin,
        f=f,
        trunk_hidden=trunk_hidden,
        trunk_depth=trunk_depth,
    ).to(DEVICE)

    model.load_state_dict(ck["model"], strict=True)
    model.eval()
    return model


@torch.no_grad()
def infer_and_plot_from_ckpt(
    ckpt_path,
    npz_path,
    split="test",
    case_index=0,
    ncols=5,
    target="T",      # temperature model
    heat_ch=0,       # <-- set heat source channel here
    q_scale=1.0,     # <-- set to 1e5 if the heat channel is dimensionless
):
    npz = np.load(npz_path, allow_pickle=True)
    X_static_np = npz["X_static"].astype(np.float32)
    times_raw = npz["times"].astype(np.float32)
    Y_T_np = npz["Y_T"].astype(np.float32)
    Y_f_np = npz["Y_f"].astype(np.float32)
    meta = npz["meta"].item() if "meta" in npz else {}

    # splits
    splits_path = Path(npz_path).parent / "splits.json"
    if splits_path.exists():
        import json
        splits = json.load(open(splits_path, "r"))
        train_ids = splits["train"]
        val_ids = splits["val"]
        test_ids = splits["test"]
    else:
        train_ids = list(range(len(X_static_np)))
        val_ids = list(range(len(X_static_np)))
        test_ids = list(range(len(X_static_np)))

    if split == "train":
        split_ids = train_ids
    elif split == "val":
        split_ids = val_ids
    elif split == "test":
        split_ids = test_ids
    else:
        raise ValueError("split must be one of: 'train', 'val', 'test'")

    # stats must be computed from TRAIN, and for the TEMPERATURE target
    stats_T = compute_train_stats(X_static_np, Y_T_np, train_ids)
    times_norm = times_raw / (float(times_raw.max()) if float(times_raw.max()) != 0.0 else 1.0)

    ds = UDeepONetCaseDataset(
        X_static_np,
        times_norm,
        Y_T_np,
        Y_f_np,
        split_ids,
        stats=stats_T,   # model target normalization (T)
        target="T",
    )
    ds.times_raw = times_raw

    model = load_model_from_ckpt(ckpt_path)

    plot_heat_trueT_predT(
        model,
        ds,
        stats_T=stats_T,
        case_idx_in_dataset=case_index,
        ncols=ncols,
        meta=meta,
        heat_ch=heat_ch,
        q_scale=q_scale,
        show_colorbar=True,
    )
    return model

In [ ]:
print(model)

In [ ]:
'''@torch.no_grad()
def plot_heat_trueT_predT(
    model,
    dataset,
    stats_T,
    case_idx_in_dataset=0,
    ncols=5,
    meta=None,
    heat_ch=0,
    q_scale=1.0,               # set to 1e5 if channel stores a dimensionless field
    heat_unit=r"W/m$^2$",
    cmap_heat="Reds",
    cmap_T="inferno",
    cmap_error="magma",
    show_colorbar=True,
):
    """
    Rows:
    0 : Heat source (static, TRUE physical values)
    1 : True Temperature
    2 : Predicted Temperature
    3 : Absolute Error in Temperature |True - Pred|

    Notes:
    - True and predicted temperature share one color scale.
    - Error has its own scale, from 0 to the max absolute error (K).
    - Heat source is drawn on its original (un-normalized) scale.
    """

    model.eval()

    sample = dataset[case_idx_in_dataset]

    # =====================================================
    # Inputs
    # =====================================================

    Xn = sample["X_static"][None].to(DEVICE)
    t = sample["times"].to(DEVICE)

    # =====================================================
    # Ground truth (raw temperature, K)
    # =====================================================

    YT_true = sample["Y_T"].cpu().numpy()

    # =====================================================
    # Prediction
    # =====================================================

    t_pred_start = time.perf_counter()

    predT_n = model(Xn, t)[0].cpu().numpy()

    prediction_time = time.perf_counter() - t_pred_start

    print(
        f"Case {case_idx_in_dataset}: "
        f"U-DeepONet prediction time = {prediction_time:.6f} s"
    )

    # Denormalize temperature
    predT = predT_n * stats_T["y_std"] + stats_T["y_mean"]

    # =====================================================
    # Temperature absolute error (K)
    # =====================================================

    errorT = np.abs(YT_true - predT)

    print(
        f"  temperature error: max = {errorT.max():.4f} K, "
        f"mean = {errorT.mean():.4f} K, "
        f"RMSE = {np.sqrt((errorT ** 2).mean()):.4f} K"
    )

    # =====================================================
    # Time indices to display
    # =====================================================

    Tn = predT.shape[0]

    idxs = np.linspace(
        0,
        Tn - 1,
        min(ncols, Tn)
    ).round().astype(int)

    # =====================================================
    # Physical extent
    # =====================================================

    extent = None

    if meta is not None and ("Lx" in meta) and ("Ly" in meta):
        extent = [
            0,
            float(meta["Lx"]),
            0,
            float(meta["Ly"]),
        ]

    # =====================================================
    # Physical times
    # =====================================================

    times_phys = getattr(dataset, "times_raw", None)

    if times_phys is None:
        times_phys = sample["times"].cpu().numpy()

    # =====================================================
    # Heat source — TRUE (physical) values
    # =====================================================
    # sample["X_static"] is normalized: (X - x_mean) / x_std.
    # Prefer the raw array held by the dataset; otherwise invert the
    # normalization using the stored per-channel statistics.

    X_static = sample["X_static"].cpu().numpy()

    if heat_ch < 0 or heat_ch >= X_static.shape[0]:
        raise ValueError(
            f"heat_ch={heat_ch} out of range "
            f"for Cin={X_static.shape[0]}"
        )

    case_id = int(sample["case_index"])

    Qmap = None

    X_raw_all = getattr(dataset, "X_static", None)

    if X_raw_all is not None:
        try:
            Qmap = np.asarray(
                X_raw_all[case_id][heat_ch],
                dtype=np.float64,
            )
        except Exception:
            Qmap = None

    if Qmap is None:
        st = getattr(dataset, "stats", None)
        if st is None:
            raise RuntimeError(
                "Cannot recover the true heat source: dataset has neither "
                "a raw X_static array nor normalization stats."
            )
        x_mean = np.asarray(st["x_mean"], dtype=np.float64).reshape(-1)[heat_ch]
        x_std = np.asarray(st["x_std"], dtype=np.float64).reshape(-1)[heat_ch]
        Qmap = X_static[heat_ch].astype(np.float64) * x_std + x_mean

    Qmap = Qmap * float(q_scale)

    print(
        f"  heat source (ch {heat_ch}) true range: "
        f"min = {Qmap.min():.6g}, max = {Qmap.max():.6g}, "
        f"mean = {Qmap.mean():.6g}"
    )

    # =====================================================
    # UNIFORM COLOR SCALES
    # =====================================================

    # ---------------- Heat source (true units) ----------------

    q_vmin = float(Qmap.min())
    q_vmax = float(Qmap.max())

    # Prevent identical limits
    if np.isclose(q_vmin, q_vmax):
        q_vmax = q_vmin + 1e-8

    # ---------------- Temperature (shared true/pred) ----------------

    T_vmin = float(min(YT_true.min(), predT.min()))
    T_vmax = float(max(YT_true.max(), predT.max()))

    if np.isclose(T_vmin, T_vmax):
        T_vmax = T_vmin + 1e-8

    # ---------------- Error ----------------

    error_vmin = 0.0
    error_vmax = float(errorT.max())

    # Prevent identical limits when the error is exactly zero
    if np.isclose(error_vmax, 0.0):
        error_vmax = 1e-8

    # =====================================================
    # Figure
    # =====================================================

    fig, axs = plt.subplots(
        4,
        len(idxs),
        figsize=(3 * len(idxs), 10),
        constrained_layout=True,
    )

    if len(idxs) == 1:
        axs = axs.reshape(4, 1)

    # =====================================================
    # Image handles
    # =====================================================

    im_heat = None
    im_trueT = None
    im_predT = None
    im_errorT = None

    # =====================================================
    # Plot
    # =====================================================

    for j, ti in enumerate(idxs):

        # =================================================
        # Row 0 — Heat Source
        # =================================================

        ax = axs[0, j]

        im_heat = ax.imshow(
            Qmap.T,
            origin="lower",
            cmap=cmap_heat,
            vmin=q_vmin,
            vmax=q_vmax,
            extent=extent,
            interpolation="nearest",
        )

        ax.set_title(
            f"Heat Source"
        )

        ax.axis("off")

        # =================================================
        # Row 1 — True Temperature
        # =================================================

        ax = axs[1, j]

        im_trueT = ax.imshow(
            YT_true[ti].T,
            origin="lower",
            cmap=cmap_T,
            vmin=T_vmin,
            vmax=T_vmax,
            extent=extent,
            interpolation="nearest",
        )

        ax.set_title(
            f"True T ({times_phys[ti]:.0f} s)"
        )

        ax.axis("off")

        # =================================================
        # Row 2 — Predicted Temperature
        # =================================================

        ax = axs[2, j]

        im_predT = ax.imshow(
            predT[ti].T,
            origin="lower",
            cmap=cmap_T,
            vmin=T_vmin,
            vmax=T_vmax,
            extent=extent,
            interpolation="nearest",
        )

        ax.set_title(
            f"Pred T ({times_phys[ti]:.0f} s)"
        )

        ax.axis("off")

        # =================================================
        # Row 3 — Absolute Error
        # =================================================

        ax = axs[3, j]

        im_errorT = ax.imshow(
            errorT[ti].T,
            origin="lower",
            cmap=cmap_error,
            vmin=error_vmin,
            vmax=error_vmax,
            extent=extent,
            interpolation="nearest",
        )

        ax.set_title(
            f"|Error| ({times_phys[ti]:.0f} s)"
        )

        ax.axis("off")

    # =====================================================
    # Colorbars
    # =====================================================

    if show_colorbar:

        from matplotlib.ticker import ScalarFormatter

        # ---------------- Heat source ----------------

        cbar = fig.colorbar(
            im_heat,
            ax=axs[0, :],
            fraction=0.025,
            pad=0.02,
        )

        cbar.set_label(
            f"Heat Source, q ({heat_unit})"
        )

        # large magnitudes (e.g. ~1e5 W/m^3) -> scientific tick labels
        _fmt = ScalarFormatter(useMathText=True)
        _fmt.set_powerlimits((-2, 3))
        cbar.formatter = _fmt
        cbar.update_ticks()

        # ---------------- True temperature ----------------

        cbar = fig.colorbar(
            im_trueT,
            ax=axs[1, :],
            fraction=0.025,
            pad=0.02,
        )

        cbar.set_label(
            "Temperature (K)"
        )

        # ---------------- Predicted temperature ----------------

        cbar = fig.colorbar(
            im_predT,
            ax=axs[2, :],
            fraction=0.025,
            pad=0.02,
        )

        cbar.set_label(
            "Temperature (K)"
        )

        # ---------------- Absolute error ----------------

        cbar = fig.colorbar(
            im_errorT,
            ax=axs[3, :],
            fraction=0.025,
            pad=0.02,
        )

        cbar.set_label(
            "Absolute Error |True - Pred| (K)"
        )

    plt.show()'''

In [ ]:
@torch.no_grad()
def plot_heat_trueT_predT(
    model,
    dataset,
    stats_T,
    case_idx_in_dataset=0,
    ncols=5,
    meta=None,
    heat_ch=0,
    q_scale=1.0,               # set to 1e5 if channel stores a dimensionless field
    heat_unit=r"W/m$^2$",
    cmap_heat="Reds",
    cmap_T="inferno",
    cmap_error="magma",
    show_colorbar=True,
    font_scale=1.0,            # 1.0 = matplotlib default; 1.4 = 40% bigger
    base_fontsize=None,        # absolute base size in pt; overrides font_scale
):
    """
    Rows:
    0 : Heat source (static, TRUE physical values)
    1 : True Temperature
    2 : Predicted Temperature
    3 : Absolute Error in Temperature |True - Pred|

    Notes:
    - True and predicted temperature share one color scale.
    - Error has its own scale, from 0 to the max absolute error (K).
    - Heat source is drawn on its original (un-normalized) scale.
    """

    model.eval()

    # =====================================================
    # Font sizes
    # =====================================================

    _base = (
        float(base_fontsize)
        if base_fontsize is not None
        else 10.0 * float(font_scale)
    )

    fs_title = 1.15 * _base      # panel titles
    fs_label = 1.15 * _base      # colorbar labels
    fs_tick = 0.95 * _base       # colorbar tick numbers

    # figure grows with the text so bigger labels do not collide
    _fig_scale = 1.0 + 0.55 * max(0.0, _base / 10.0 - 1.0)

    sample = dataset[case_idx_in_dataset]

    # =====================================================
    # Inputs
    # =====================================================

    Xn = sample["X_static"][None].to(DEVICE)
    t = sample["times"].to(DEVICE)

    # =====================================================
    # Ground truth (raw temperature, K)
    # =====================================================

    YT_true = sample["Y_T"].cpu().numpy()

    # =====================================================
    # Prediction
    # =====================================================

    t_pred_start = time.perf_counter()

    predT_n = model(Xn, t)[0].cpu().numpy()

    prediction_time = time.perf_counter() - t_pred_start

    print(
        f"Case {case_idx_in_dataset}: "
        f"U-DeepONet prediction time = {prediction_time:.6f} s"
    )

    # Denormalize temperature
    predT = predT_n * stats_T["y_std"] + stats_T["y_mean"]

    # =====================================================
    # Temperature absolute error (K)
    # =====================================================

    errorT = np.abs(YT_true - predT)

    mae_T = float(errorT.mean())
    rmse_T = float(np.sqrt((errorT ** 2).mean()))

    ss_res = float(((YT_true - predT) ** 2).sum())
    ss_tot = float(((YT_true - YT_true.mean()) ** 2).sum())

    if ss_tot > 0.0:
        r2_T = 1.0 - ss_res / ss_tot
    else:
        r2_T = float("nan")

    print(
        f"  temperature error: max = {errorT.max():.4f} K, "
        f"MAE = {mae_T:.4f} K, "
        f"RMSE = {rmse_T:.4f} K, "
        f"R2 = {r2_T:.4f}"
    )

    # =====================================================
    # Time indices to display
    # =====================================================

    Tn = predT.shape[0]

    idxs = np.linspace(
        0,
        Tn - 1,
        min(ncols, Tn)
    ).round().astype(int)

    # =====================================================
    # Physical extent
    # =====================================================

    extent = None

    if meta is not None and ("Lx" in meta) and ("Ly" in meta):
        extent = [
            0,
            float(meta["Lx"]),
            0,
            float(meta["Ly"]),
        ]

    # =====================================================
    # Physical times
    # =====================================================

    times_phys = getattr(dataset, "times_raw", None)

    if times_phys is None:
        times_phys = sample["times"].cpu().numpy()

    # =====================================================
    # Heat source — TRUE (physical) values
    # =====================================================
    # sample["X_static"] is normalized: (X - x_mean) / x_std.
    # Prefer the raw array held by the dataset; otherwise invert the
    # normalization using the stored per-channel statistics.

    X_static = sample["X_static"].cpu().numpy()

    if heat_ch < 0 or heat_ch >= X_static.shape[0]:
        raise ValueError(
            f"heat_ch={heat_ch} out of range "
            f"for Cin={X_static.shape[0]}"
        )

    case_id = int(sample["case_index"])

    Qmap = None

    X_raw_all = getattr(dataset, "X_static", None)

    if X_raw_all is not None:
        try:
            Qmap = np.asarray(
                X_raw_all[case_id][heat_ch],
                dtype=np.float64,
            )
        except Exception:
            Qmap = None

    if Qmap is None:
        st = getattr(dataset, "stats", None)
        if st is None:
            raise RuntimeError(
                "Cannot recover the true heat source: dataset has neither "
                "a raw X_static array nor normalization stats."
            )
        x_mean = np.asarray(st["x_mean"], dtype=np.float64).reshape(-1)[heat_ch]
        x_std = np.asarray(st["x_std"], dtype=np.float64).reshape(-1)[heat_ch]
        Qmap = X_static[heat_ch].astype(np.float64) * x_std + x_mean

    Qmap = Qmap * float(q_scale)

    print(
        f"  heat source (ch {heat_ch}) true range: "
        f"min = {Qmap.min():.6g}, max = {Qmap.max():.6g}, "
        f"mean = {Qmap.mean():.6g}"
    )

    # =====================================================
    # UNIFORM COLOR SCALES
    # =====================================================

    # ---------------- Heat source (true units) ----------------

    q_vmin = float(Qmap.min())
    q_vmax = float(Qmap.max())

    # Prevent identical limits
    if np.isclose(q_vmin, q_vmax):
        q_vmax = q_vmin + 1e-8

    # ---------------- Temperature (shared true/pred) ----------------

    T_vmin = float(min(YT_true.min(), predT.min()))
    T_vmax = float(max(YT_true.max(), predT.max()))

    if np.isclose(T_vmin, T_vmax):
        T_vmax = T_vmin + 1e-8

    # ---------------- Error ----------------

    error_vmin = 0.0
    error_vmax = float(errorT.max())

    # Prevent identical limits when the error is exactly zero
    if np.isclose(error_vmax, 0.0):
        error_vmax = 1e-8

    # =====================================================
    # Figure
    # =====================================================

    fig, axs = plt.subplots(
        4,
        len(idxs),
        figsize=(
            3 * _fig_scale * len(idxs),
            10 * _fig_scale,
        ),
        constrained_layout=True,
    )

    if len(idxs) == 1:
        axs = axs.reshape(4, 1)

    # =====================================================
    # Image handles
    # =====================================================

    im_heat = None
    im_trueT = None
    im_predT = None
    im_errorT = None

    # =====================================================
    # Plot
    # =====================================================

    for j, ti in enumerate(idxs):

        # =================================================
        # Row 0 — Heat Source
        # =================================================

        ax = axs[0, j]

        im_heat = ax.imshow(
            Qmap.T,
            origin="lower",
            cmap=cmap_heat,
            vmin=q_vmin,
            vmax=q_vmax,
            extent=extent,
            interpolation="nearest",
        )

        ax.set_title(
            "Heat Source",
            fontsize=fs_title,
        )

        ax.axis("off")

        # =================================================
        # Row 1 — True Temperature
        # =================================================

        ax = axs[1, j]

        im_trueT = ax.imshow(
            YT_true[ti].T,
            origin="lower",
            cmap=cmap_T,
            vmin=T_vmin,
            vmax=T_vmax,
            extent=extent,
            interpolation="nearest",
        )

        ax.set_title(
            f"True T ({times_phys[ti]:.0f} s)",
            fontsize=fs_title,
        )

        ax.axis("off")

        # =================================================
        # Row 2 — Predicted Temperature
        # =================================================

        ax = axs[2, j]

        im_predT = ax.imshow(
            predT[ti].T,
            origin="lower",
            cmap=cmap_T,
            vmin=T_vmin,
            vmax=T_vmax,
            extent=extent,
            interpolation="nearest",
        )

        ax.set_title(
            f"Pred T ({times_phys[ti]:.0f} s)",
            fontsize=fs_title,
        )

        ax.axis("off")

        # =================================================
        # Row 3 — Absolute Error
        # =================================================

        ax = axs[3, j]

        im_errorT = ax.imshow(
            errorT[ti].T,
            origin="lower",
            cmap=cmap_error,
            vmin=error_vmin,
            vmax=error_vmax,
            extent=extent,
            interpolation="nearest",
        )

        ax.set_title(
            f"|Error| ({times_phys[ti]:.0f} s)",
            fontsize=fs_title,
        )

        ax.axis("off")

    # =====================================================
    # Colorbars
    # =====================================================

    if show_colorbar:

        from matplotlib.ticker import ScalarFormatter

        # ---------------- Heat source ----------------

        cbar = fig.colorbar(
            im_heat,
            ax=axs[0, :],
            fraction=0.025,
            pad=0.02,
        )

        cbar.set_label(
            f"Heat Source, q ({heat_unit})",
            fontsize=fs_label,
        )

        cbar.ax.tick_params(labelsize=fs_tick)

        # large magnitudes (e.g. ~1e5 W/m^3) -> scientific tick labels
        _fmt = ScalarFormatter(useMathText=True)
        _fmt.set_powerlimits((-2, 3))
        cbar.formatter = _fmt
        cbar.update_ticks()

        # the 'x10^4' offset text is not covered by tick_params
        cbar.ax.yaxis.get_offset_text().set_fontsize(fs_tick)

        # ---------------- True temperature ----------------

        cbar = fig.colorbar(
            im_trueT,
            ax=axs[1, :],
            fraction=0.025,
            pad=0.02,
        )

        cbar.set_label(
            "Temperature (K)",
            fontsize=fs_label,
        )

        cbar.ax.tick_params(labelsize=fs_tick)

        # ---------------- Predicted temperature ----------------

        cbar = fig.colorbar(
            im_predT,
            ax=axs[2, :],
            fraction=0.025,
            pad=0.02,
        )

        cbar.set_label(
            "Temperature (K)",
            fontsize=fs_label,
        )

        cbar.ax.tick_params(labelsize=fs_tick)

        # ---------------- Absolute error ----------------

        cbar = fig.colorbar(
            im_errorT,
            ax=axs[3, :],
            fraction=0.025,
            pad=0.02,
        )

        cbar.set_label(
            "Absolute Error (K)",
            fontsize=fs_label,
        )

        cbar.ax.tick_params(labelsize=fs_tick)

    plt.show()

In [ ]:
# =====================================================================
#  NORMALIZATION FIX
#  Replaces infer_and_plot_from_ckpt in cell 18 of BOTH notebooks
#  (Temp and Liqfrac). Works for target="T" and target="f".
#
#  The bug: stats were computed from the npz being evaluated, so on any
#  dataset other than the training one, inputs are scaled differently
#  than during training and outputs are de-scaled with the wrong
#  y_mean / y_std.
#
#  The fix: always derive x_mean, x_std, y_mean, y_std and t_max from
#  the dataset the model was TRAINED on.
# =====================================================================

import json as _json


def load_train_norm(train_npz_path, target=TARGET, verbose=True):
    """
    Recover the normalization constants and time scale from the training
    dataset. Call once, reuse across all cases.
    """
    train_npz_path = Path(train_npz_path)
    npz = np.load(train_npz_path, allow_pickle=True)

    X = npz["X_static"].astype(np.float32)
    times_raw = npz["times"].astype(np.float32)
    Y = npz["Y_T" if target == "T" else "Y_f"].astype(np.float32)

    splits_path = train_npz_path.parent / "splits.json"

    if splits_path.exists():
        train_ids = _json.load(open(splits_path, "r"))["train"]
    else:
        train_ids = list(range(len(X)))
        print("  WARNING: no splits.json beside the training npz; using all "
              "cases. Stats will not exactly match training.")

    stats = compute_train_stats(X, Y, train_ids)

    norm = {
        "stats": stats,
        "t_max": float(times_raw.max()),
        "Cin": int(X.shape[1]),
        "HW": (int(X.shape[2]), int(X.shape[3])),
        "n_train": len(train_ids),
        "target": target,
    }

    if verbose:
        print(f"Train norm ({train_npz_path.parent.name}, target='{target}'): "
              f"{norm['n_train']} cases, Cin={norm['Cin']}, "
              f"grid={norm['HW']}, t_max={norm['t_max']:.0f} s")
        print(f"  y_mean={stats['y_mean']:.6f}  y_std={stats['y_std']:.6f}")
        print(f"  q ch0 : x_mean={float(stats['x_mean'].reshape(-1)[0]):.6g}  "
              f"x_std={float(stats['x_std'].reshape(-1)[0]):.6g}")

    return norm


@torch.no_grad()
def infer_and_plot_from_ckpt(
    ckpt_path,
    npz_path,
    train_npz_path=None,   # npz the model was TRAINED on
    norm=None,             # or a cached load_train_norm() result
    split="test",
    case_index=0,
    ncols=5,
    target="T",
    heat_ch=0,
    q_scale=1.0,
    font_scale=1.0,
):
    npz = np.load(npz_path, allow_pickle=True)
    X_static_np = npz["X_static"].astype(np.float32)
    times_raw = npz["times"].astype(np.float32)
    Y_T_np = npz["Y_T"].astype(np.float32)
    Y_f_np = npz["Y_f"].astype(np.float32)
    meta = npz["meta"].item() if "meta" in npz else {}

    # ---------------- split ids of the dataset being evaluated ----------
    splits_path = Path(npz_path).parent / "splits.json"

    if splits_path.exists():
        splits = _json.load(open(splits_path, "r"))
        ids_by_split = {
            "train": splits["train"],
            "val": splits["val"],
            "test": splits["test"],
        }
    else:
        allids = list(range(len(X_static_np)))
        ids_by_split = {"train": allids, "val": allids, "test": allids}

    ids_by_split["all"] = list(range(len(X_static_np)))

    if split not in ids_by_split:
        raise ValueError("split must be 'train', 'val', 'test' or 'all'")

    split_ids = ids_by_split[split]

    # =====================================================
    # THE FIX: normalization from the TRAINING dataset
    # =====================================================

    if norm is None:
        if train_npz_path is None:
            raise ValueError(
                "Pass train_npz_path (or norm=) so the model's own "
                "normalization is used. Without it the stats come from "
                "the evaluated dataset and predictions are biased."
            )
        norm = load_train_norm(train_npz_path, target=target)

    if norm["target"] != target:
        raise ValueError(
            f"norm was built for target='{norm['target']}' but you asked "
            f"for '{target}'. Build one per target."
        )

    if norm["Cin"] != X_static_np.shape[1]:
        raise ValueError(
            f"Cin mismatch: training {norm['Cin']} vs evaluated "
            f"{X_static_np.shape[1]}."
        )

    stats = norm["stats"]
    t_max = norm["t_max"] if norm["t_max"] != 0.0 else 1.0

    # Scale times by the TRAINING t_max, not this dataset's.
    times_norm = times_raw / t_max

    ds = UDeepONetCaseDataset(
        X_static_np,
        times_norm,
        Y_T_np,
        Y_f_np,
        split_ids,
        stats=stats,
        target=target,
    )

    # UDeepONetCaseDataset.__init__ sets t_scale = max(times) and divides
    # again in __getitem__. times_norm is already scaled, so neutralize it
    # or the training t_max would be silently undone.
    ds.t_scale = 1.0

    ds.times_raw = times_raw

    if float(times_raw.max()) > norm["t_max"] * 1.001:
        print(f"  NOTE: this dataset runs to {float(times_raw.max()):.0f} s, "
              f"past the training horizon of {norm['t_max']:.0f} s "
              f"(normalized t up to "
              f"{float(times_raw.max()) / norm['t_max']:.2f}): extrapolation.")

    model = load_model_from_ckpt(ckpt_path)

    if target == "T":
        plot_heat_trueT_predT(
            model, ds,
            stats_T=stats,
            case_idx_in_dataset=case_index,
            ncols=ncols, meta=meta, heat_ch=heat_ch,
            q_scale=q_scale,
            show_colorbar=True,
        )
    else:
        plot_trueT_trueF_predF_with_heat(
            model, ds,
            stats_f=stats,
            case_idx_in_dataset=case_index,
            ncols=ncols, meta=meta, heat_ch=heat_ch,
            q_scale=q_scale,
            show_colorbar=True,
        )

    return model

### CROSS GEOMETRY FOR CIRCLE

In [ ]:
TRAIN_NPZ = r"runs/dataset_run_20260902-114515/udeeponet_dataset.npz"
NEW_NPZ   = r"runs/dataset_run_20260902-115529/udeeponet_dataset.npz"
CKPT      = r"runs/dataset_run_20260902-114515/checkpoints_udeeponet_T/best.pt"

norm = load_train_norm(TRAIN_NPZ, target=TARGET)

for i in range(15):
    infer_and_plot_from_ckpt(
        ckpt_path=CKPT, npz_path=NEW_NPZ, norm=norm,
        split="all", case_index=i, target=TARGET, ncols=6,
    )

In [ ]:
TRAIN_NPZ = r"runs/dataset_run_20260817-133248/udeeponet_dataset.npz"
NEW_NPZ   = r"runs/dataset_run_20260902-115529/udeeponet_dataset.npz"
CKPT      = r"runs/dataset_run_20260817-133248/checkpoints_udeeponet_T/best.pt"

norm = load_train_norm(TRAIN_NPZ, target=TARGET)

for i in range(15):
    infer_and_plot_from_ckpt(
        ckpt_path=CKPT, npz_path=NEW_NPZ, norm=norm,
        split="all", case_index=i, target=TARGET, ncols=6,
    )